In [1]:
import torch

inputs = torch.tensor( # 6개의 토큰이 들어왔고, 각 토큰 당 3차원 벡터를 가짐
    [[0.43, 0.15, 0.89], #Your (x^1)
    [0.55, 0.87, 0.66], #journey (x^2)
    [0.57, 0.85, 0.64], #starts (x^3)
    [0.22, 0.58, 0.33], #with (x^4)
    [0.77, 0.25, 0.10], #one (x^5)
    [0.05, 0.80, 0.55]] #step (x^6)
)

query = inputs[1] # 두 번째 입력 토큰을 쿼리 토큰으로 사용
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs): 
    attn_scores_2[i] = torch.dot(query, x_i) # 각 입력(행)에 대해 어텐션 스코어 계산, 두 번째 단어가 i번째 단어를 얼마나 중요하게 생각하는가?
print(attn_scores_2) # ex. attn_scores_[0]은 두 번째 입력 토큰에 대한 첫 번째 입력 토큰과의 문맥 관계 점수

ModuleNotFoundError: No module named 'torch'

In [ ]:
res = 0
for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]
print(res)
print(torch.dot(inputs[0], query))
# 점곱 연산 : 1. 두 벡터를 결합하여 하나의 스칼라 값을 만듦
#            2. 두 벡터가 얼마나 가까이 놓여 있는지 알 수 있는 유사도

tensor(0.9544)
tensor(0.9544)


In [ ]:
# 정규화 목적 : 1)어텐션 가중치의 합이 1이 되도록 하기 위해서. 2) 해석하기 용이, 3) LLM을 훈련할 때 안정성을 유지하는 데 도움
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("어텐션 가중치: ",attn_weights_2_tmp)
print("합 : ", attn_weights_2_tmp.sum())

어텐션 가중치:  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
합 :  tensor(1.0000)


In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("어텐션 가중치:",attn_weights_2_naive)
print("합", attn_weights_2_naive.sum())

attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("어텐션 가중치: ",attn_weights_2)
print("합: ", attn_weights_2.sum())


어텐션 가중치: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
합 tensor(1.)
어텐션 가중치:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
합:  tensor(1.)


In [ ]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape) #(3,)
for i,x_i in enumerate(inputs):
    print(f"attn_wieghts_2[{i}] : {attn_weights_2[i] :.4f} \n x_i : {x_i}")
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2) #문맥 벡터의 n번째 원소는, 모든 입력 단어의 n번째 원소들을 어텐션 가중치로 가중 평균낸 값

attn_wieghts_2[0] : 0.1385 
 x_i : tensor([0.4300, 0.1500, 0.8900])
attn_wieghts_2[1] : 0.2379 
 x_i : tensor([0.5500, 0.8700, 0.6600])
attn_wieghts_2[2] : 0.2333 
 x_i : tensor([0.5700, 0.8500, 0.6400])
attn_wieghts_2[3] : 0.1240 
 x_i : tensor([0.2200, 0.5800, 0.3300])
attn_wieghts_2[4] : 0.1082 
 x_i : tensor([0.7700, 0.2500, 0.1000])
attn_wieghts_2[5] : 0.1581 
 x_i : tensor([0.0500, 0.8000, 0.5500])
tensor([0.4419, 0.6515, 0.5683])


In [ ]:
attn_scores = torch.empty(6,6)
for i,x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i,j] = torch.dot(x_i,x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
attn_scores = inputs @ inputs.T
print(attn_scores)
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [ ]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("두 번쨰 행의 합: ", row_2_sum)
print("모든 행의 합 :", attn_weights.sum(dim=-1))

두 번쨰 행의 합:  1.0
모든 행의 합 : tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [ ]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)
print("이전에 계산한 두 번째 문맥 벡터 : ", context_vec_2)


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
이전에 계산한 두 번째 문맥 벡터 :  tensor([0.4419, 0.6515, 0.5683])


In [ ]:
# 훈련 가능한 가중치를 가진 셀프 어텐션 메커니즘
x_2 = inputs[1]
d_in = inputs.shape[1] # 입력 임베딩 크기 d_in=3
d_out = 2 # 출력 임베딩 크기 d_out=2

torch.manual_seed(123)
w_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False) # 출력을 간단하게 하기 위해서 False, 실제 모델 훈련 시에는 True로 지정
w_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
w_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

query_2 = x_2 @ w_query # (3) * (3,2)
key_2 = x_2 @ w_key
value_2 = x_2 @ w_value
print(query_2)
print(key_2)
print(value_2)

tensor([0.4306, 1.4551])
tensor([0.4433, 1.1419])
tensor([0.3951, 1.0037])


In [ ]:
keys = inputs @ w_key # (6,3) * (3,2)
values = inputs @ w_value
print("keys.shape", keys.shape)
print("values.shape", values.shape)

keys.shape torch.Size([6, 2])
values.shape torch.Size([6, 2])


In [ ]:
keys_2 = keys[1] #(2)
attn_score_22 = query_2.dot(keys_2) # (2) (2)의 점곱 = 스칼라
print(attn_score_22)

attn_scores_2 = query_2 @ keys.T # (2) * (6,2)T, 주어진 쿼리에 대한 모든 어텐션 점수
print(attn_scores_2)

tensor(1.8524)
tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [ ]:
d_k = keys.shape[-1] #(2)

attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [ ]:
context_vec_2 = attn_weights_2 @ values #(6,) * (6,2), 어텐션 가중치와 각각의 값(Value) 벡터의 곱
print(attn_weights_2,"\n",values)
print(context_vec_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820]) 
 tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]])
tensor([0.3061, 0.8210])


In [ ]:
# Self attention mechanism

import torch.nn as nn

class SelfAttention_v1(nn.Module): # nn.Module을 사용하면 모델에 추가된 층과 파라미터를 자동으로 관리하고 foward() 메소드를 오버라이드하여 정방향 계산을 정의할 수 있음
    def __init__(self, d_in, d_out):
        super().__init__()
        self.w_query = nn.Parameter(torch.rand((d_in,d_out))) # nn.parameter로 감싸면, torch에서 "이건 나중에 오차 역전파를 통해 값을 계산하고 업데이트해야 할 가중치구나"를 인식하게 됨
        self.w_key = nn.Parameter(torch.rand((d_in,d_out)))
        self.w_value = nn.Parameter(torch.rand((d_in,d_out)))

    def forward(self, x):
        keys = x @ self.w_key
        queries = x @ self.w_query
        values = x @ self.w_value

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in,d_out)
print(sa_v1(inputs))



tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False): # Linear층은 편향 유닛(bias unit)을 사용하지 않으면 행렬 곱셈과 동일한 연산 수해, 최적화된 가중치 초기화 방법 사용할 수 있어 모델 훈련을 안정적이고 효율적으로 만들어줌
        super().__init__()
        self.w_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_value = nn.Linear(d_in, d_out, bias=qkv_bias)


    def forward(self, x):
        keys = self.w_key(x)
        queries = self.w_query(x)
        values = self.w_value(x)

        attn_scores = queries @ keys.T # keys.transpose(-2, -1)
        attn_weights = torch.softmax(
            attn_scores/keys.shape[-1]**0.5, dim = -1
        )
        context_vec = attn_weights @ values
        return context_vec
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in,d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [ ]:
print(sa_v1.w_query)
print(sa_v1.w_key)
print(sa_v1.w_value)

print(sa_v2.w_query.weight)
print(sa_v2.w_key.weight)
print(sa_v2.w_value.weight)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True)
Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]], requires_grad=True)
Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]], requires_grad=True)
Parameter containing:
tensor([[ 0.3161,  0.4568,  0.5118],
        [-0.1683, -0.3379, -0.0918]], requires_grad=True)
Parameter containing:
tensor([[ 0.4058, -0.4704,  0.2368],
        [ 0.2134, -0.2601, -0.5105]], requires_grad=True)
Parameter containing:
tensor([[ 0.2526, -0.1415, -0.1962],
        [ 0.5191, -0.0852, -0.2043]], requires_grad=True)


In [ ]:
'''
with torch.no_grad():
    sa_v1.w_query.copy_(sa_v2.w_query.weight.T)
    sa_v1.w_value.copy_(sa_v2.w_value.weight.T)
    sa_v1.w_key.copy_(sa_v2.w_key.weight.T)
print(sa_v1.w_query)
print(sa_v1.w_value)
print(sa_v1.w_key)

torch.manual_seed(123)
print(sa_v1(inputs))
'''

'\nwith torch.no_grad():\n    sa_v1.w_query.copy_(sa_v2.w_query.weight.T)\n    sa_v1.w_value.copy_(sa_v2.w_value.weight.T)\n    sa_v1.w_key.copy_(sa_v2.w_key.weight.T)\nprint(sa_v1.w_query)\nprint(sa_v1.w_value)\nprint(sa_v1.w_key)\n\ntorch.manual_seed(123)\nprint(sa_v1(inputs))\n'

In [ ]:
'''
#3.1 문제 정답
sa_v1.w_query = torch.nn.Parameter(sa_v2.w_query.weight.T)
sa_v1.w_key = torch.nn.Parameter(sa_v2.w_key.weight.T)
sa_v1.w_value = torch.nn.Parameter(sa_v2.w_value.weight.T)
sa_v1(inputs)
'''

'\n#3.1 문제 정답\nsa_v1.w_query = torch.nn.Parameter(sa_v2.w_query.weight.T)\nsa_v1.w_key = torch.nn.Parameter(sa_v2.w_key.weight.T)\nsa_v1.w_value = torch.nn.Parameter(sa_v2.w_value.weight.T)\nsa_v1(inputs)\n'

In [ ]:
queries = sa_v2.w_query(inputs)
keys = sa_v2.w_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim = -1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
context_length = attn_scores.shape[0] 
mask_simple = torch.tril(torch.ones(context_length, context_length)) # torch.tril : 주대각선 기준 위쪽 부분 0으로 만듦
print(mask_simple)


tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [ ]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [ ]:
row_sums = masked_simple.sum(dim=-1, keepdim=True) #False로 할 시, shape가 (6,), True일 시 (6,1)
print(row_sums.shape)
print(row_sums)
print(masked_simple.shape)
print(masked_simple)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

torch.Size([6, 1])
tensor([[0.1921],
        [0.3700],
        [0.5357],
        [0.6775],
        [0.8415],
        [1.0000]], grad_fn=<SumBackward1>)
torch.Size([6, 6])
tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1) #triu는 tril와 반대로 주대각선 아래를 0으로 만듦
                                                                          #diagonal=1 : 주대각선 바로 한 칸 위쪽부터 남김
print(mask)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf) # 첫 번쨰 인자에서 True인 부분을 두 번째 인자 값으로 변경
print(masked)


tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [ ]:
attn_weights = torch.softmax(masked / keys.shape[1]**0.5, dim = 1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6,6)
print(dropout(example)) # 삭제된 값을 남은 원소들로 보상하기 위해 행렬에서 남은 원소의 값을 1/1-dropout_ratio배로 늘림


tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [ ]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


In [ ]:
batch = torch.stack((inputs, inputs), dim=0) # 0번째 차원에 input을 2개 쌓음
print(batch.shape)
print(batch)

torch.Size([2, 6, 3])
tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])


In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, 
                qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.w_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer( # 지정한 텐서를 CPU/GPU로 자동으로 이동, 장치가 동일하지 않다는 오류 피함, 업데이트 X
            "mask",
            torch.triu(torch.ones(context_length,context_length),
            diagonal=1)
        )
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.w_key(x)
        queries = self.w_query(x)
        values = self.w_value(x)

        attn_scores = queries @ keys.transpose(1,2) # 두 번쨰 차원(num_tokens)과 세 번쨰 차원(d_in) 바꿈
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim = -1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values

        return context_vec
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape", context_vecs.shape)

context_vecs.shape torch.Size([2, 6, 2])


In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(
                d_in, d_out, context_length, dropout, qkv_bias
            )
            for _ in range(num_heads)]
        )
    
    def forward(self,x):
        return torch.cat([head(x) for head in self.heads], dim=-1) # head(x) = __call__로, forward 실행

torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in,d_out,context_length,0.0,num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape", context_vecs.shape) 

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape torch.Size([2, 6, 4])


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads==0), \
            "d_out은 num_heads로 나누어 떨어져야 합니다"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # 원하는 출력 차원에 맞도록 투영 차원을 낮춤
        self.w_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_key = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.w_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out,d_out) # Linear 층을 사용하여 헤드의 출력 결합
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self,x):
        b, num_tokens, d_in = x.shape
        keys = self.w_key(x) # Tensor shape : (b, num_tokens, d_out)
        queries = self.w_query(x)
        values = self.w_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries= queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1,2) # (b, num_tokens, num_heads, head_dim) → (b, num_heads, num_tokens, head_dim)으로 변경
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        attn_scores = queries @ keys.transpose(2,3) # 각 헤드에 대해 점곱 수행
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens] # 토큰 개수로 마스크를 자름

        attn_scores.masked_fill_(mask_bool, -torch.inf) # 마스크를 사용해 어텐션 점수를 채움

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        ) 
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1,2) # Tensor shape : (b, num_tokens, num_heads, head_dim)
        context_vec = context_vec.contiguous().view(b,num_tokens,self.d_out) # 헤드 결합. self.d_out = self.num_heads * self.head_dim
        context_vec = self.out_proj(context_vec) 
        return context_vec

In [ ]:
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573], # (b, num_heads, num_tokens, head_dim) = (1,2,3,4)
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],
                    [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2,3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


In [ ]:
first_head = a[0, 0, :,  :]
first_res = first_head @ first_head.T
print("첫 번째 헤드:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print('\n 두 번째 헤드:\n', second_res)

첫 번째 헤드:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

 두 번째 헤드:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


In [ ]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)


tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


In [ ]:
#연습문제 2
#연습문제 3